# Gold Layer — Top Products by Category

Identifies the top 3 best-selling products in each product category, ranked by units sold. Uses a window function over `silver.order_items` joined with `silver.orders` and `silver.products`.

## Design Decisions

* **Metric is `COUNT(*)` of order_items rows** — each row represents one unit sold, so units sold equals the row count after the join.
* **DENSE_RANK over RANK or ROW_NUMBER** — units_sold has integer ties, and DENSE_RANK preserves ties without skipping ranks. Two products tied at rank 3 both appear; the next product would be rank 4. The table can therefore contain more than 3 rows per category when ties occur.
* **Excludes Unknown category** — `Unknown` represents products with missing or untranslatable categories from the Silver layer's coalesce fallback. Ranking products within Unknown is not business-meaningful.
* **Status filter on orders** — excludes canceled and unavailable orders, consistent with the rest of the Gold layer.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%run ../Utilities/silver_dq_checks

# Silver Layer — Data Quality Checks

This notebook defines reusable data quality check functions used across the Silver layer transformation notebooks.

Each function validates a specific property of a transformed DataFrame before it is written to a Silver Delta table. On success the function prints a PASSED message; on failure it raises a `ValueError` with details, stopping the pipeline.

## Checks Defined Here

* `check_not_null` — verifies that specified columns have no null or blank values
* `check_unique` — verifies that the given key (single or composite) has no duplicates
* `check_row_count` — verifies that the Silver row count falls within an expected percentage range of the Bronze source

## How To Use

Import these functions into a Silver transformation notebook by running:
​```
%run ../Utilities/silver_dq_checks
​```
Then call the functions on the transformed DataFrame before writing to Silver.

### check_not_null
Validates that the specified columns contain no null or blank/whitespace values.

### check_unique
Validates that the specified key columns produce unique rows. Supports single-column or composite keys.

### check_row_count
Validates that the Silver row count is within an acceptable percentage range of the Bronze source row count.


### check_event_sequence
Validates that timestamp columns appear in the expected chronological order. Skips rows where either timestamp in a pair is null. Raises on any violation.

### identify_event_sequence_violations
Adds one boolean column per consecutive timestamp pair to mark whether each row violates the expected order. Returns the annotated DataFrame without raising — callers use it to split clean rows from violations.

### check_value_range
Validates that values in specified columns meet a minimum threshold rule (e.g. `price > 0`, `freight_value >= 0`). Each rule is a dict specifying column, min_value, and inclusivity. Raises on any violation, reporting all rules that failed.

In [0]:
catalog_name = 'retail_lakehouse'
bronze_schema = 'bronze'
silver_schema = 'silver'
gold_schema = 'gold'
quarantine_schema = 'quarantine'

### Top Products By Category

In [0]:

df_orders = spark.read.table(f'{catalog_name}.{silver_schema}.orders')\
    .filter(~col('order_status').isin(['canceled', 'unavailable']))\
        .select('order_id')

df_orders.createOrReplaceTempView('orders')
                                                                                                                                                                                                                                  
df_products = spark.read.table(f'{catalog_name}.{silver_schema}.products').filter(col('category_name') != 'Unknown').select('product_id','category_name')
df_products.createOrReplaceTempView('products')

df_order_items = spark.read.table(f'{catalog_name}.{silver_schema}.order_items').select('order_id','product_id')
df_order_items.createOrReplaceTempView('order_items')

In [0]:
top_products_df = spark.sql(f'''
WITH units_sold  AS 
(Select p.category_name,p.product_id,count(*) as units_sold
FROM orders o inner join order_items t on o.order_id = t.order_id
inner join products p on t.product_id = p.product_id
group by p.category_name,p.product_id)

, product_rank AS
(Select category_name, product_id, units_sold,DENSE_RANK() OVER(PARTITION BY category_name ORDER BY units_sold DESC) as r
from units_sold)

Select category_name, product_id, units_sold, r as rank_in_category 
from product_rank 
where r <= 3''')

In [0]:
check_not_null(top_products_df, ['category_name', 'product_id', 'units_sold', 'rank_in_category'])
check_unique(top_products_df, ['category_name', 'product_id'])
check_value_range(top_products_df, [
    {'column': 'units_sold', 'min_value': 0, 'inclusive': False},
    {'column': 'rank_in_category', 'min_value': 1, 'inclusive': True}
])

check_not_null: PASSED
check_unique: PASSED
check_value_range: PASSED


### Write to gold schema

In [0]:
top_products_df.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.{gold_schema}.top_products_by_category')